# **FOREWORD**

This is a simple replica of the public kernels that are doing stacking with linear + xgb models.

These kernels are doing **linear models on 5-fold CV and boosted tree on 7-fold CV causing a data leakage**. My version corrects this problem and slightly improves the score with a fold trick. 

Here, we keep **original as columns** and replicate the process with the same model

# **IMPORTS**

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import * 
from sklearn.metrics import *
from sklearn.preprocessing import *
from sklearn.pipeline import *
from sklearn.impute import *

import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
# SYNTAX CHECK FLAG - FALSE FOR LB SUBMISSION
test_req = False

# **ANCILLARY CODE**

In [3]:
def preprocess(df):
    df_temp = df.copy()

    df_temp['study_hours_squared'] = df_temp['study_hours'] ** 2
    df_temp['study_hours_cubed'] = df_temp['study_hours'] ** 3
    df_temp['class_attendance_squared'] = df_temp['class_attendance'] ** 2
    df_temp['sleep_hours_squared'] = df_temp['sleep_hours'] ** 2
    df_temp['age_squared'] = df_temp['age'] ** 2

    df_temp['log_study_hours'] = np.log1p(df_temp['study_hours'])
    df_temp['log_class_attendance'] = np.log1p(df_temp['class_attendance'])
    df_temp['log_sleep_hours'] = np.log1p(df_temp['sleep_hours'])
    df_temp['sqrt_study_hours'] = np.sqrt(df_temp['study_hours'])
    df_temp['sqrt_class_attendance'] = np.sqrt(df_temp['class_attendance'])

    # Interaction features
    df_temp['study_hours_times_attendance'] = df_temp['study_hours'] * df_temp['class_attendance']
    df_temp['study_hours_times_sleep'] = df_temp['study_hours'] * df_temp['sleep_hours']
    df_temp['attendance_times_sleep'] = df_temp['class_attendance'] * df_temp['sleep_hours']

    # Ratio features (add small epsilon to avoid division by zero)
    eps = 1e-5
    df_temp['study_hours_over_sleep'] = df_temp['study_hours'] / (df_temp['sleep_hours'] + eps)
    df_temp['attendance_over_sleep'] = df_temp['class_attendance'] / (df_temp['sleep_hours'] + eps)

    # Encode categorical variables to numeric ordinal values
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_rating_map = {'low': 0, 'medium': 1, 'high': 2}
    exam_difficulty_map = {'easy': 0, 'medium': 1, 'hard': 2}

    df_temp['sleep_quality_numeric'] = df_temp['sleep_quality'].map(sleep_quality_map).fillna(1).astype(int)
    df_temp['facility_rating_numeric'] = df_temp['facility_rating'].map(facility_rating_map).fillna(1).astype(int)
    df_temp['exam_difficulty_numeric'] = df_temp['exam_difficulty'].map(exam_difficulty_map).fillna(1).astype(int)

    # Interaction between encoded categoricals and key numeric features
    df_temp['study_hours_times_sleep_quality'] = df_temp['study_hours'] * df_temp['sleep_quality_numeric']
    df_temp['attendance_times_facility'] = df_temp['class_attendance'] * df_temp['facility_rating_numeric']
    df_temp['sleep_hours_times_difficulty'] = df_temp['sleep_hours'] * df_temp['exam_difficulty_numeric']
    df_temp['age_times_study_hours'] = df_temp['age'] * df_temp['study_hours']
    df_temp['age_times_attendance'] = df_temp['age'] * df_temp['class_attendance']

    # Composite feature: learning efficiency
    df_temp['efficiency'] = (df_temp['study_hours'] * df_temp['class_attendance']) / (df_temp['sleep_hours'] + 1)

    numeric_features = [
        # 'feature_formula',
        'study_hours_squared', 'study_hours_cubed',
        'class_attendance_squared', 'sleep_hours_squared', 'age_squared',
        'log_study_hours', 'log_class_attendance', 'log_sleep_hours',
        'sqrt_study_hours', 'sqrt_class_attendance',
        'study_hours_times_attendance', 'study_hours_times_sleep',
        'attendance_times_sleep', 'study_hours_over_sleep',
        'attendance_over_sleep',
        'sleep_quality_numeric', 'facility_rating_numeric', 'exam_difficulty_numeric',
        'study_hours_times_sleep_quality', 'attendance_times_facility',
        'sleep_hours_times_difficulty', 'age_times_study_hours',
        'age_times_attendance', 'efficiency']

    return df_temp[base_features + numeric_features]

In [4]:
xgb_params = {
    'eval_metric'          : 'rmse',
    'n_estimators'         : 10000 if test_req == False else 100,
    'learning_rate'        : 0.007,
    'max_depth'            : 7,
    'subsample'            : 0.8,
    'reg_lambda'           : 3,
    'colsample_bytree'     : 0.6,
    'colsample_bynode'     : 0.7,
    'tree_method'          : 'hist',
    'random_state'         : 42,
    'early_stopping_rounds': 500 if test_req == False else 10,
    'enable_categorical'   : True,
    'device'               : 'cuda'
}

# CORRECTION FROM PUBLIC KERNEL WITH IMPROVEMENT
FOLDS = 15 if test_req == False else 3
kf    = KFold(n_splits=FOLDS, shuffle=True, random_state=1003)

TARGET = 'exam_score'

# **MODEL PIPELINE**

In [5]:

train_file    = "/kaggle/input/playground-series-s6e1/train.csv"
test_file     = "/kaggle/input/playground-series-s6e1/test.csv"
original_file = "/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv"
train_df      = pd.read_csv(train_file)
test_df       = pd.read_csv(test_file)
original_df   = pd.read_csv(original_file)
submission_df = pd.read_csv("/kaggle/input/playground-series-s6e1/sample_submission.csv")
base_features = [col for col in train_df.columns if col not in [TARGET, 'id']]
CATS          = train_df.select_dtypes('object').columns.to_list()

min_tgt       = train_df[TARGET].min()
max_tgt       = train_df[TARGET].max()

X_raw      = preprocess(train_df)
y          = train_df[TARGET].reset_index(drop=True)
X_test_raw = preprocess(test_df)
full_data  = pd.concat([X_raw, X_test_raw], axis=0)

numeric_cols = [
    'study_hours_squared', 
    'study_hours_cubed',
    'class_attendance_squared', 
    'sleep_hours_squared', 
    'age_squared',
    'log_study_hours', 
    'log_class_attendance', 
    'log_sleep_hours',
    'sqrt_study_hours', 
    'sqrt_class_attendance',
    'study_hours_times_attendance', 
    'study_hours_times_sleep',
    'attendance_times_sleep', 
    'study_hours_over_sleep',
    'attendance_over_sleep',
    'sleep_quality_numeric', 
    'facility_rating_numeric',
    'exam_difficulty_numeric',
    'study_hours_times_sleep_quality', 
    'attendance_times_facility',
    'sleep_hours_times_difficulty', 
    'age_times_study_hours',
    'age_times_attendance',
    'efficiency',
]

print(f"---> Adding original as columns")
for col in base_features :
    df_ = original_df.groupby(col).agg({TARGET : ["mean", "count", "max", "min"]})
    df_.columns = [f"Omean{col}", f"Ocount{col}", f"Omax{col}", f"Omin{col}"]
    numeric_cols.extend(df_.columns.tolist())
    full_data = full_data.merge(df_.reset_index(), on = [col], how = "left")
    del df_
    
for col in numeric_cols:
    full_data[col] = full_data[col].astype(float)

X               = full_data.iloc[0 : -len(test_df)].copy()
X_test          = full_data.iloc[-len(test_df):].copy()
N_SAMPLES_TRAIN = X.shape[0]
N_SAMPLES_TEST  = X_test.shape[0]
oof_pred_lr     = np.zeros(N_SAMPLES_TRAIN)
test_preds_lr   = np.zeros((N_SAMPLES_TEST, FOLDS))

fold_rmse_lr    = []
lr_models       = []

print(f"\n---> Beginning linear model pipeline")
for fold, (train_index, val_index) in enumerate(kf.split(X, y), start=1):

    X_train_fold, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val = y.iloc[train_index], y.iloc[val_index]

    X_train_combined = X_train_fold
    y_train_combined = y_train_fold

    target_encoder  = TargetEncoder(smooth='auto', target_type='continuous')
    X_train_encoded = X_train_combined.copy()
    X_val_encoded   = X_val.copy()
    X_test_encoded  = X_test.copy()

    X_train_encoded[CATS] = target_encoder.fit_transform(X_train_combined[CATS], y_train_combined)
    X_val_encoded[CATS]   = target_encoder.transform(X_val[CATS])
    X_test_encoded[CATS]  = target_encoder.transform(X_test[CATS])

    lr_model = make_pipeline(*[SimpleImputer(strategy = "mean"), StandardScaler(), LinearRegression()])
    lr_model.fit(X_train_encoded, y_train_combined.ravel())
    lr_models.append(lr_model)

    lr_val_pred   = lr_model.predict(X_val_encoded)
    lr_test_pred  = lr_model.predict(X_test_encoded)   
    lr_val_pred   = np.clip(lr_val_pred,   a_min = min_tgt,  a_max = max_tgt)
    lr_test_pred  = np.clip(lr_test_pred,  a_min = min_tgt,  a_max = max_tgt)
  
    rmse_lr                    = root_mean_squared_error(y_val, lr_val_pred)
    oof_pred_lr[val_index]     = lr_val_pred
    test_preds_lr[:, fold - 1] = lr_test_pred

    print(f"---> Score = {rmse_lr:.8f} | Fold {fold}")
    fold_rmse_lr.append(rmse_lr)

for col in base_features:
    full_data[col] = full_data[col].astype(str)
    full_data[col] = full_data[col].astype('category')

for col in numeric_cols:
    full_data[col] = full_data[col].astype(float)

X          = full_data.iloc[0 : -len(test_df)].copy()
X_test     = full_data.iloc[-len(test_df):].copy()

X['feature_lr_pred']          = oof_pred_lr
X_test['feature_lr_pred']     = test_preds_lr.mean(axis=1)

test_predictions = []
oof_predictions  = np.zeros(len(X))

print(f"\n---> Beginning XgBoost pipeline")
for fold, (train_index, val_index) in enumerate(kf.split(X, y), start = 1):

    X_train_fold, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val = y.iloc[train_index], y.iloc[val_index]

    X_train_combined = X_train_fold
    y_train_combined = y_train_fold

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_train_combined, 
        y_train_combined, 
        eval_set=[(X_val, y_val)], 
        verbose= 0
    )

    val_preds                  = model.predict(X_val)
    oof_predictions[val_index] = val_preds
    score                      = np.sqrt(mean_squared_error(y_val, val_preds))
    print(f"---> Score = {score:.8f} | Fold {fold}")

    test_preds = model.predict(X_test)
    test_predictions.append(test_preds)

oof_rmse = root_mean_squared_error(y, oof_predictions)
print(f"\n---> Overall score = {oof_rmse:.8f}")

oof_df = pd.DataFrame({'id': train_df['id'], TARGET: oof_predictions})
oof_df.to_csv('xgb_oof.csv', index=False)

submission_df[TARGET] = np.mean(test_predictions, axis=0)
submission_df.to_csv('submission.csv', index=False)

---> Adding original as columns

---> Beginning linear model pipeline
---> Score = 8.85673285 | Fold 1
---> Score = 8.82905868 | Fold 2
---> Score = 8.88648975 | Fold 3
---> Score = 8.80833357 | Fold 4
---> Score = 8.77016027 | Fold 5
---> Score = 8.86228353 | Fold 6
---> Score = 8.83059296 | Fold 7
---> Score = 8.82838177 | Fold 8
---> Score = 8.85577915 | Fold 9
---> Score = 8.82153560 | Fold 10
---> Score = 8.87025788 | Fold 11
---> Score = 8.76103349 | Fold 12
---> Score = 8.84682886 | Fold 13
---> Score = 8.82366802 | Fold 14
---> Score = 8.85934216 | Fold 15

---> Beginning XgBoost pipeline
---> Score = 8.62708879 | Fold 1
---> Score = 8.60049447 | Fold 2
---> Score = 8.67725758 | Fold 3
---> Score = 8.58340893 | Fold 4
---> Score = 8.56312818 | Fold 5
---> Score = 8.65063066 | Fold 6
---> Score = 8.60553412 | Fold 7
---> Score = 8.61101760 | Fold 8
---> Score = 8.64415090 | Fold 9
---> Score = 8.59035652 | Fold 10
---> Score = 8.64301869 | Fold 11
---> Score = 8.53558801 | Fold 

# **SUBMISSION**

In [6]:
submission_df[TARGET] = np.mean(test_predictions, axis=0)
submission_df.to_csv('submission.csv', index=False)

!ls
print()
!head submission.csv

__notebook__.ipynb  submission.csv  xgb_oof.csv

id,exam_score
630000,68.984795
630001,68.79395
630002,90.72811
630003,56.61719
630004,47.01991
630005,70.806076
630006,72.2506
630007,59.654564
630008,80.288864
